# Training the Simplified ABMAP Model

This notebook mirrors the end-to-end workflow for fitting the main model
included in this repository.  It relies only on the files shipped with
the project so it can run in an offline environment.

## 1. Configure the Python path

When the notebook is opened from the `notebooks/` directory the project
root is one level above.  The following cell adds that directory to the
Python module search path so modules such as `abmap` can be imported
without installing them as a package.

In [ ]:
import os
import sys

def find_project_root(start_dir):
    """Walk upwards from *start_dir* until the repository root is found."""
    current = os.path.abspath(start_dir)
    while True:
        if os.path.exists(os.path.join(current, "abmap")):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError("Unable to locate project root")
        current = parent

PROJECT_ROOT = find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root resolved to: {PROJECT_ROOT}")

## 2. Inspect the full dataset

The synthetic dataset distributed with the repository lives in
`data/main_dataset.csv`.  Each row contains eight engineered features
and a binary label that indicates whether the antibody-antigen pair is
considered a binder.

In [ ]:
from abmap.data import load_csv_dataset

dataset_path = os.path.join(PROJECT_ROOT, "data", "main_dataset.csv")
features, labels, header = load_csv_dataset(dataset_path)

print(f"Loaded {len(features)} samples with {len(header) - 1} features each.")
print(f"First feature columns: {header[:-1]}")
print(f"Label column: {header[-1]}")

## 3. Train the main model

`train_logistic_regression` performs batch gradient descent on the full
dataset while tracking loss and accuracy on both the training and
validation splits.  The hyper-parameters below mirror those used in the
command-line script.

In [ ]:
from abmap.training import train_logistic_regression

model, history = train_logistic_regression(
    features,
    labels,
    val_ratio=0.25,
    epochs=30,
    batch_size=128,
    learning_rate=0.2,
    seed=2023,
)

print(f"Training finished after {len(history)} epochs.")

## 4. Review metrics

The recorded history exposes the loss and accuracy for each epoch on
both data splits.  The final metrics provide a concise summary of the
model's performance.

In [ ]:
import json

print("Last 5 epochs:")
for record in history[-5:]:
    print(json.dumps(record, indent=2, sort_keys=True))

final_metrics = history[-1]
print("
Final metrics:")
print(json.dumps(final_metrics, indent=2, sort_keys=True))

## 5. Persist artifacts

Saving the model and training history ensures reproducibility and keeps
the notebook aligned with the command-line workflow.  The resulting
files appear in the `artifacts/` directory.

In [ ]:
from abmap.training import save_training_artifacts

model_path = os.path.join(PROJECT_ROOT, "artifacts", "main_model_from_notebook.json")
history_path = os.path.join(PROJECT_ROOT, "artifacts", "training_history_from_notebook.json")
save_training_artifacts(model, history, model_path=model_path, history_path=history_path)

print(f"Model saved to: {model_path}")
print(f"History saved to: {history_path}")